# HW4: Reading the Verdicts

**COMPSS 211A | Fall 2026 | Student copy**

In this homework you use TF-IDF on about 3,800 posts from Reddit's r/AmItheAsshole to ask which kinds of situations the community judges harshly. You will choose which posts to keep, remove junk words from the TF-IDF results, and compare posts with different verdicts. Last, you label 50 texts from your project data and use the same comparison to see which words go with each of your labels.

**Due:** Sunday, November 15, 2026 at 11:59 p.m.

Fill in every cell marked "Your turn" and answer the written questions. When a cell asks for a prediction, write your guess before you run anything. Keep the variable names we ask for, because we look for them when grading.

## The data

[r/AmItheAsshole](https://www.reddit.com/r/AmItheAsshole/) ("AITA") is a Reddit forum where people describe a conflict and ask whether they were in the wrong. Commenters reply with a verdict, and the post gets a label (a *flair*) based on the top-voted comment:

| Flair | Short form | Meaning |
| --- | --- | --- |
| Not the A-hole | NTA | The poster was not in the wrong. |
| Asshole | YTA | "You're the asshole": the poster was in the wrong. |
| Everyone Sucks | ESH | Everyone involved behaved badly. |
| No A-holes here | NAH | Nobody behaved badly. |

The file `data/aita_top_subs.csv` has 5,000 posts from 2018 to 2021. They are among the most upvoted posts on the forum, so they are not a random sample. The file has no usernames, but these are real people's stories: don't try to find out who wrote them, and don't quote them outside this class.

## What you will practice

- Deciding which rows to keep.
- Predicting what cleaning code does.
- Building TF-IDF features (Week 8).
- Removing junk words by changing the settings.
- Comparing two groups of posts.
- Labeling texts from your project data, and comparing the labels with TF-IDF.

## Helpful references

- The Week 8 reference (`lessons/week08_nlp-fundamentals/nlp_features_reference.md`) explains `TfidfVectorizer` and its settings.
- Lab 8 has the first versions of `clean_comment` and `rank_terms`.
- A word gets a high TF-IDF score in a post when it appears in that post but is rare in other posts. A high score means the word is distinctive, not that it is important.
- In the TF-IDF matrix, each row is one post, in the same order as your table, and each column is one word.

## AI and collaboration policy

You may use AI to help you understand an error message, as long as you say so in a note in your notebook. Write the code and the written answers yourself. In Task 6 you may share your file of project texts with your teammates, but label the texts yourself.

In [ ]:
from pathlib import Path
import json
import sys

import re
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Decide which posts to keep

Some rows can't be used. Some posts were removed or deleted, so the text says `[removed]` or `[deleted]`. Some flairs are not verdicts (`UPDATE`, `META`), and some posts have no flair. The cell below counts each problem.

In [ ]:
VERDICT_LABELS = {
    "Not the A-hole": "NTA",
    "not the a-hole": "NTA",
    "Asshole": "YTA",
    "Everyone Sucks": "ESH",
    "No A-holes here": "NAH",
}

raw_posts = pd.read_csv(course_data_path("aita_top_subs.csv"))
print(len(raw_posts), "posts in the file")
print(raw_posts["selftext"].isin(["[removed]", "[deleted]"]).sum(), "posts removed or deleted")

raw_posts["flair_text"].value_counts(dropna=False)

**Your turn:** make a table called `posts` with only the rows where:

- `flair_text` is one of the keys in `VERDICT_LABELS` (hint: `.isin(...)`), and
- `selftext` is not missing and is not `[removed]` or `[deleted]` (hint: `~` turns True into False and False into True).

Add `.copy()` at the end of the filter. Then add two columns:

- `verdict`: the short form (NTA, YTA, ESH, NAH). Use `.map(VERDICT_LABELS)`.
- `text`: the title and the post joined with a space: `posts["title"] + " " + posts["selftext"]`.

Last, show how many posts you kept and the share of each verdict (`value_counts(normalize=True)`).

In [ ]:
# Your turn
# 1. Filter raw_posts into posts, and add the verdict and text columns.
posts = ...

# 2. Show the number of posts and the share of each verdict.


## 2. Predict what the cleaning does

Here is `clean_comment` from Lab 8, renamed `clean_post`. Read it line by line.

`re.sub(pattern, replacement, text)` replaces every part of `text` that matches `pattern`. In `[^a-z0-9\s'-]`, the square brackets mean "any one of these characters", and the `^` means "any character *not* in this set". So the first `re.sub` replaces everything except lowercase letters, digits, spaces, apostrophes, and hyphens.

In [ ]:
def clean_post(text):
    """Lowercase the text, remove most punctuation, and tidy up spaces."""
    lowered = str(text).lower()
    # Replace anything that isn't a-z, 0-9, a space, an apostrophe, or a hyphen with a space.
    no_punctuation = re.sub(r"[^a-z0-9\s'-]", " ", lowered)
    # Turn runs of spaces into a single space, and trim both ends.
    return re.sub(r"\s+", " ", no_punctuation).strip()

**Your turn:** before you run anything, type what you think `clean_post` returns for each string below. Then run the cell. It prints `correct` or `different` for each guess, and the actual result.

Example: `"AITA for skipping my sister's wedding?"` becomes `"aita for skipping my sister's wedding"`.

In [ ]:
my_predictions = {
    "WIBTA if I told my MIL to leave?": "...",
    "My fiancé (27F) said NO.": "...",
    "EDIT: ok, I'm the a**hole": "...",
}

for original, guess in my_predictions.items():
    actual = clean_post(original)
    result = "correct" if guess == actual else "different"
    print(f"{result:9} | {original!r} -> {actual!r}")

In [ ]:
# Now clean every post. .apply() runs clean_post once for each row.
posts["clean_text"] = posts["text"].apply(clean_post)
posts[["text", "clean_text"]].head()

### Your response


Answer in two or three sentences:

1. Which prediction did you get wrong, if any? What did you misread in the code?
2. What happened to `fiancé`, and which line caused it? How many posts contain `é`? (Try `posts["text"].str.contains("é").sum()`.)


> Write your response here, then delete this line.

## 3. Build TF-IDF

`rank_terms` below is the Lab 8 version with one more column. `mean_tfidf` is a word's average score across all posts. `n_posts` is the number of posts that contain the word. A word can have a high average because it is in many posts, or because it scores very high in a few. `n_posts` tells you which.

In [ ]:
def rank_terms(vectorizer, matrix, n=25):
    """Return the n words with the highest average TF-IDF score."""
    # Average score of each word (column) across all posts (rows).
    average_scores = np.asarray(matrix.mean(axis=0)).ravel()
    # Number of posts (rows) that contain each word (column).
    posts_with_word = np.asarray((matrix > 0).sum(axis=0)).ravel()

    ranked = pd.DataFrame({
        "term": vectorizer.get_feature_names_out(),
        "mean_tfidf": average_scores,
        "n_posts": posts_with_word,
    })
    return ranked.sort_values("mean_tfidf", ascending=False).head(n).reset_index(drop=True)

**Your turn:**

1. Write down three words you expect near the top of the list.
2. Make `tfidf_vectorizer` with `TfidfVectorizer(stop_words="english", min_df=5)`. `stop_words="english"` drops common words like "the" and "is". `min_df=5` drops words that are in fewer than five posts, such as typos and names.
3. Make `tfidf_matrix` with `tfidf_vectorizer.fit_transform(posts["clean_text"])`. Print its shape.
4. Make `top_terms` with `rank_terms(tfidf_vectorizer, tfidf_matrix)`.

In [ ]:
# 1. My three predicted top words:
#

# Your turn: steps 2 to 4
tfidf_vectorizer = ...
tfidf_matrix = ...
top_terms = ...


## 4. Clean up the vocabulary

Look at `top_terms`. Some of the top words say little about the stories:

- `aita`: almost every title starts with "AITA for...".
- `don`, `didn`, `ve`: `TfidfVectorizer` splits words at apostrophes, so "don't" becomes `don` and `t`. The `t` is dropped because it is one letter.
- `fianc`, from Task 2.

You can add your own words to the stop-word list. `|` joins two sets:

```python
custom_stop_words = list(ENGLISH_STOP_WORDS | {"aita", "wibta"})
```

**Your turn:**

1. Make `custom_stop_words` with the words you want to remove. Look past the top 25 with `rank_terms(tfidf_vectorizer, tfidf_matrix, n=60)`. Note that removing `don` and `didn` removes "don't" and "didn't" from your features.
2. Make `custom_vectorizer`, `custom_matrix`, and `top_terms_custom` as in Task 3, but with `stop_words=custom_stop_words`. Print the shape of `custom_matrix`.
3. Run the cell to see the top words before and after.

In [ ]:
# Your turn
custom_stop_words = ...
custom_vectorizer = ...
custom_matrix = ...
top_terms_custom = ...

term_comparison = pd.DataFrame({
    "before": top_terms["term"],
    "after": top_terms_custom["term"],
})
term_comparison

## 5. Compare YTA and NTA posts

Do posts judged YTA talk about different things than posts judged NTA?

Common words like `told` and `said` are in every kind of post, so the top words for each verdict look almost the same. `compare_groups` below looks at the difference instead: for each word, its average score in group a minus its average score in group b. Words in `leans_a` score higher in group a, and words in `leans_b` score higher in group b.

To pick the posts with one verdict, use a True/False mask:

```python
is_yta = (posts["verdict"] == "YTA").to_numpy()
```

Use `posts`, `custom_vectorizer`, and `custom_matrix` together, so the rows line up.

In [ ]:
def compare_groups(vectorizer, matrix, mask_a, mask_b, n=15):
    """Return the words whose average score differs most between two groups of rows."""
    average_a = np.asarray(matrix[mask_a].mean(axis=0)).ravel()
    average_b = np.asarray(matrix[mask_b].mean(axis=0)).ravel()
    difference = pd.DataFrame({
        "term": vectorizer.get_feature_names_out(),
        "difference": average_a - average_b,
    }).sort_values("difference", ascending=False)
    return pd.DataFrame({
        "leans_a": difference["term"].head(n).to_numpy(),
        "leans_b": difference["term"].tail(n)[::-1].to_numpy(),
    })

**Your turn:**

1. Make `is_yta` and `is_nta`. Call `compare_groups` with YTA as group a and NTA as group b. Save the result as `verdict_terms`.
2. Pick one word from either column. Show the `title` of five posts with that verdict whose `clean_text` contains the word (hint: `.str.contains(...)` and a verdict condition). Read two of the full posts.

In [ ]:
# Your turn 1: masks and verdict_terms
is_yta = ...
is_nta = ...
verdict_terms = ...


# Your turn 2: five titles that contain your word



### Your response


Answer in three or four sentences:

1. What does your word tell you about the situations in that group? Use the posts you read.
2. Which words did you remove in Task 4, and why?
3. What can't you conclude from this comparison? Think about who wrote these posts and who voted on them.


> Write your response here, then delete this line.

## 6. Label your project texts

In Lab 7 your team wrote labels for your project data. Here you label more texts on your own, and use `compare_groups` from Task 5 to see which words go with one of your labels. In HW5 you check Gemini against the labels you make here.

**Before you start:**

1. Take 50 texts at random from your project data that were not in Lab 7 (for example, `your_table.sample(50, random_state=1)`). If one text is very long, such as a speech or a filing, use one paragraph as one text. Leave out texts with names, usernames, or other details that identify a person.
2. Save them as `project_texts.csv`, in the same folder as this notebook, with the columns `id` and `text`.
3. Open the file in Excel or Google Sheets, add a column `label`, and give each text one of your team's labels from the end of Lab 7. Save it as a CSV file again. If your data already has a label, such as a star rating or a flair, use that column as `label` instead.

**Your turn:**

1. Fill in `PROJECT_LABELS` and run the cell. It loads the file, checks every label, and counts the texts with each label.
2. Add a column `clean_text` to `project_texts` with `clean_post`.
3. Make `project_vectorizer` with `TfidfVectorizer(stop_words="english", min_df=2)`, and `project_matrix` from `project_texts["clean_text"]`. `min_df=2` keeps words that are in at least two texts; five would drop most words in only 50 texts.
4. Pick the label you care most about. Make `label_terms` with `compare_groups`: the texts with that label as group a, and all other texts as group b.

In [ ]:
PROJECT_LABELS = ["...", "...", "unclear"]  # your team's labels from the end of Lab 7

project_texts = pd.read_csv("project_texts.csv", dtype={"id": str})
assert "..." not in PROJECT_LABELS, "Fill in PROJECT_LABELS."
assert project_texts["label"].isin(PROJECT_LABELS).all(), "Every label must be one of PROJECT_LABELS."
print(len(project_texts), "texts")
project_texts["label"].value_counts()

In [ ]:
# Your turn: steps 2 to 4
project_vectorizer = ...
project_matrix = ...
label_terms = ...

### Your response


Answer in two or three sentences:

1. Which words lean toward your label? Do they fit your definition, or do they point to something else, such as the topic of the texts or how long they are?
2. With only 50 texts, one long text can push a word up the list. Check one word from `leans_a`: in how many texts does it appear (hint: `project_texts["clean_text"].str.contains(...).sum()`)?


> Write your response here, then delete this line.